In [19]:
from moviepy.editor import VideoFileClip
import os
import azure.cognitiveservices.speech as speechsdk
from moviepy.editor import AudioFileClip
import time
import json
import tiktoken

In [6]:
# Función para extraer el audio de un video 
def convertir_mp4_a_wav(video_path, output_audio_path):
    # Cargar el video
    video = VideoFileClip(video_path)
    # Extraer el audio del video
    audio = video.audio
    # Guardar el audio en formato WAV con la máxima calidad
    audio.write_audiofile(output_audio_path, codec='pcm_s16le')
    # Liberar recursos
    audio.close()
    video.close()
    print(f"El audio ha sido extraído y guardado en {output_audio_path}")

In [7]:
def convertir_mp3_a_wav(mp3_path, output_wav_path):
    try:
        # Cargar el archivo MP3
        audio = AudioFileClip(mp3_path)
        
        # Guardar el archivo en formato WAV
        audio.write_audiofile(output_wav_path, codec='pcm_s16le')
        
        print(f"El archivo ha sido convertido a WAV y guardado en {output_wav_path}")
    except Exception as e:
        print(f"Ocurrió un error: {e}")

In [8]:
def seconds_to_hms(seconds):
    # Obtener horas, minutos y segundos 
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = seconds % 60
    # Convertir al formato de HH:MM:SS
    return f"{hours:02d}:{minutes:02d}:{seconds:02.0f}"

In [9]:
def ExtraccionTexto( output_audio_path,json_path,speech_key,service_region):


    #Configuración del servicio
    speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=service_region)
    #Establecer el idioma
    speech_config.speech_recognition_language = "es-ES"

    #Inicialización y configuración del reconocimiento de audio
    audio_config = speechsdk.audio.AudioConfig(filename=output_audio_path)
    speech_recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)

    #Almacen de resultados 
    results_with_timestamps = {}
    #Varibale de control para determinar cuando finaliza el proceso de reconocimiento
    done = False

    # Función para extraer fragmentos de texto y sus timestamps 
    def recognized_handler(evt):
        #Se activa cada vez que el servicio de reconocimiento de voz identifica y transcribe un segmento de voz.
        if evt.result.reason == speechsdk.ResultReason.RecognizedSpeech:
            #Inicio y duración del segmento reconocido (1 tick = 100 nanoseconds)
            offset_seconds = evt.result.offset / 10000000
            duration_seconds = evt.result.duration / 10000000
            #Inicio y fin del segmento reconocido
            start_time = offset_seconds
            end_time = offset_seconds + duration_seconds
            # Convertir a formato hh:mm:ss
            start_time_formatted = seconds_to_hms(start_time)
            end_time_formatted = seconds_to_hms(end_time)
            # Almacenar el texto con su tiempo correspondiente
            time_key = f"{start_time_formatted} - {end_time_formatted}"
            results_with_timestamps[time_key] = evt.result.text
    
    # Función para cerrar la sesión de reconocimiento
    def session_stopped_handler(evt): #"evt" representa el evento que desencadena la llamada a la función
        #Indicar que la variable "done" es de la función general y no de la presente función
        nonlocal done
        print("La sesión de extracción de texto ha terminado.")
        #Cambio de valor para señalar que el reconocimiento ha terminado
        done = True

    #Conecta el evento recognized con el manejador recognized_handler. Esto significa que cada vez que el servicio de reconocimiento 
    #de voz reconoce y transcribe un segmento de voz correctamente, se llama automáticamente a la función recognized_handler. 
    speech_recognizer.recognized.connect(recognized_handler)

    #Conecta el evento session_stopped con el manejador session_stopped_handler. Este evento se dispara cuando la sesión de 
    #reconocimiento de voz se detiene
    speech_recognizer.session_stopped.connect(session_stopped_handler)

    #Inicia el reconocimiento continuo de voz. Permite que el servicio procese audio continuamente en tiempo real o de un archivo, 
    #generando eventos de transcripción a medida que identifica el habla.
    speech_recognizer.start_continuous_recognition()    

    #Función para mantener el programa en un bucle de espera activa mientras se está realizando el reconocimiento de voz 
    #en segundo plano.
    while not done:
        time.sleep(0.5)

    #Detener formalmente el reconocimiento continuo.Esto asegura que los recursos se liberan correctamente y que cualquier 
    #proceso de limpieza necesario se realiza.
    speech_recognizer.stop_continuous_recognition()

    # Guardar los resultados en un archivo JSON
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(results_with_timestamps, f, ensure_ascii=False, indent=4)
        print("La extracción de texto fue guardado en: ",json_path)

In [20]:
pip install newrelic


     ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
     -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
     ------------------------- -------------- 0.8/1.2 MB 2.0 MB/s eta 0:00:01
     ---------------------------------------- 1.2/1.2 MB 2.1 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: still running...
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: still running...
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for newrelic: filename=newrelic-10.6.0-py3-none-any.whl size=750350 sha256=d